In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from datetime import datetime

Fetch API

In [0]:
url="https://opensky-network.org/api/states/all"

response=requests.get(url)

data=response.json()

In [0]:
print(data)

Convert to DataFrame

In [0]:
columns=[
"icao24",
"callsign",
"origin_country",
"time_position",
"last_contact",
"longitude",
"latitude",
"baro_altitude",
"on_ground",
"velocity",
"true_track",
"vertical_rate",
"sensors",
"geo_altitude",
"squawk",
"spi",
"position_source"
]

pdf=pd.DataFrame(data["states"],columns=columns)

In [0]:
print(pdf)

Add Metadata

In [0]:
pdf["ingestion_timestamp"]=datetime.now()

pdf["load_date"]=datetime.now().date()

Convert to Spark

In [0]:
spark_df=spark.createDataFrame(pdf)

Convert NULL column to String

In [0]:
from pyspark.sql.functions import col

spark_df = spark_df.withColumn(
    "sensors",
    col("sensors").cast("string")
)

Write CSV to ADLS

In [0]:
path="abfss://batch@uralibootcamp.dfs.core.windows.net/raw/Flights"

spark_df.write \
.mode("append") \
.option("header","true") \
.csv(path)

In [0]:
%sql
SHOW STORAGE CREDENTIALS;

In [0]:
%sql
LIST 'abfss://batch@uralibootcamp.dfs.core.windows.net/'